# SLM Code Documentation — Dataset Creation Pipeline
**Target:** Qwen2.5-Coder-1.5B-Instruct fine-tuning via QLoRA  
**Environment:** Kaggle Free Tier (T4 GPU, 29GB RAM, 20GB Disk)  
**Estimated Runtime:** ~75 minutes  
**Output:** Cleaned, balanced, chat-formatted dataset saved to `/kaggle/working/`

### Pipeline Stages
1. Install dependencies
2. Load + process Tier 1 datasets (code2doc, CodeXGLUE, self-oss-instruct)
3. Stream + filter Tier 2 datasets (CodeSearchNet, docstring_corpus)
4. Merge, deduplicate, balance
5. Format to Qwen2.5 chat template
6. Save outputs + print statistics

## Cell 1 — Install Dependencies

In [ ]:
%%capture
!pip install datasets transformers huggingface_hub rouge_score --quiet
print("✅ Dependencies installed")

## Cell 2 — Imports & Configuration

In [ ]:
import os
import re
import ast
import json
import hashlib
import random
import warnings
from pathlib import Path
from collections import defaultdict, Counter
from typing import Optional

import datasets
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets
from transformers import AutoTokenizer

warnings.filterwarnings("ignore")
datasets.logging.set_verbosity_error()

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)

# ── Output paths ───────────────────────────────────────────────────────────────
OUTPUT_DIR  = Path("/kaggle/working/slm_docgen_dataset")
CACHE_DIR   = Path("/kaggle/working/hf_cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"]           = str(CACHE_DIR)
os.environ["HF_DATASETS_CACHE"] = str(CACHE_DIR)

# ── Dataset size caps (tune to stay within 20GB disk + 29GB RAM) ───────────────
CAPS = {
    "code2doc":          35_000,
    "codexglue":         30_000,   # 10K per language
    "self_oss_instruct": 40_000,
    "codesearchnet":     60_000,   # 20K per language
    "docstring_corpus":  20_000,
}

# ── Language targets (after balancing) ─────────────────────────────────────────
LANGUAGE_TARGET = {"python": 0.40, "java": 0.35, "javascript": 0.25}
FINAL_DATASET_SIZE = 150_000   # target after dedup + balance

# ── Quality thresholds ─────────────────────────────────────────────────────────
MIN_CODE_LINES   = 3
MIN_DOC_TOKENS   = 8
MAX_DOC_TOKENS   = 600
MIN_QUALITY      = 0.50

# ── Tokenizer (for max_length checks during formatting) ───────────────────────
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
MAX_SEQ_LENGTH = 2048   # training context window

print("✅ Configuration loaded")
print(f"   Output dir  : {OUTPUT_DIR}")
print(f"   Target size : {FINAL_DATASET_SIZE:,} samples")
print(f"   Languages   : {LANGUAGE_TARGET}")

## Cell 3 — Core Utility Functions

In [ ]:
# ── Language normalisation ─────────────────────────────────────────────────────
LANG_MAP = {
    "python": "python", "py": "python",
    "java": "java",
    "javascript": "javascript", "js": "javascript",
    "typescript": "javascript", "ts": "javascript",
}

def normalize_language(raw: str) -> Optional[str]:
    if not raw:
        return None
    return LANG_MAP.get(raw.lower().strip())


# ── Style inference ────────────────────────────────────────────────────────────
def infer_style(doc: str, language: str) -> str:
    """Heuristic documentation style detector."""
    if language == "java":
        return "javadoc"
    if language == "javascript":
        return "jsdoc"
    # Python style patterns
    if re.search(r":param\s+\w+:|:type\s+\w+:|:returns?:|:rtype:", doc):
        return "restructuredtext"
    if re.search(r"Args:\s*\n\s+\w+", doc):
        return "google"
    if re.search(r"Parameters\s*\n\s*[-─]{3,}", doc):
        return "numpy"
    return "plain"


# ── Code block extractor (for instruction-format datasets) ─────────────────────
def extract_code_block(text: str) -> Optional[str]:
    """Extract first fenced code block from markdown text."""
    match = re.search(r"```(?:python|java|javascript|js|\w+)?\n([\s\S]*?)```", text)
    return match.group(1).strip() if match else None


# ── Deduplication hash ─────────────────────────────────────────────────────────
_seen_hashes: set = set()

def hash_code(code: str) -> str:
    normalised = re.sub(r'\s+', ' ', code.strip().lower())
    return hashlib.md5(normalised.encode()).hexdigest()

def is_duplicate(code: str) -> bool:
    h = hash_code(code)
    if h in _seen_hashes:
        return True
    _seen_hashes.add(h)
    return False

def reset_dedup():
    _seen_hashes.clear()


# ── Quality filters ────────────────────────────────────────────────────────────
BOILERPLATE_PATTERNS = [
    r"^auto[- ]generated", r"^generated by", r"^todo[:\s]",
    r"^fixme[:\s]", r"^not implemented", r"@inheritdoc",
    r"^\s*\{\s*@inheritDoc\s*\}\s*$",
]

def is_boilerplate(doc: str) -> bool:
    doc_lower = doc.lower().strip()
    for pattern in BOILERPLATE_PATTERNS:
        if re.search(pattern, doc_lower):
            return True
    return False

def is_trivial_doc(code: str, doc: str) -> bool:
    """True if doc is just a restatement of the function name."""
    match = re.search(r"def (\w+)|public\s+\w+\s+(\w+)\s*\(|function\s+(\w+)|const\s+(\w+)\s*=", code)
    if not match:
        return False
    name = next(g for g in match.groups() if g).lower()
    name_words = re.sub(r'([A-Z])', r' \1', name).lower().strip()
    doc_clean = doc.lower().strip().rstrip(".")
    # Check if doc is essentially just the function name restated
    return doc_clean in (name, name_words, name.replace("_", " "))

def passes_quality_filter(sample: dict) -> bool:
    """Master quality gate — returns True if sample should be kept."""
    if sample is None:
        return False

    code = sample.get("code", "").strip()
    doc  = sample.get("target_doc", "").strip()
    lang = sample.get("language", "")

    # Presence checks
    if not code or not doc or not lang:
        return False

    # Language whitelist
    if lang not in {"python", "java", "javascript"}:
        return False

    # Length checks
    if len(code.splitlines()) < MIN_CODE_LINES:
        return False
    doc_tokens = doc.split()
    if len(doc_tokens) < MIN_DOC_TOKENS or len(doc_tokens) > MAX_DOC_TOKENS:
        return False

    # Quality score check
    if sample.get("quality_score", 1.0) < MIN_QUALITY:
        return False

    # Content quality checks
    if is_boilerplate(doc):
        return False
    if is_trivial_doc(code, doc):
        return False

    return True


print("✅ Utility functions defined")

## Cell 4 — Dataset Adapters
Each adapter normalises a raw dataset record into the unified schema.

In [ ]:
# ── Unified schema ─────────────────────────────────────────────────────────────
# {
#   "code":          str,
#   "target_doc":    str,
#   "language":      str,   # python | java | javascript
#   "style":         str,   # google | numpy | restructuredtext | javadoc | jsdoc | plain
#   "quality_score": float, # 0.0–1.0
#   "source":        str,   # dataset tag
# }

def adapt_code2doc(example: dict) -> Optional[dict]:
    lang = normalize_language(example.get("language", ""))
    if not lang:
        return None
    doc = (
        example.get("documentation")
        or example.get("docstring")
        or example.get("doc", "")
    ).strip()
    return {
        "code":          example.get("code", "").strip(),
        "target_doc":    doc,
        "language":      lang,
        "style":         example.get("style") or infer_style(doc, lang),
        "quality_score": float(example.get("quality_score", 0.7)),
        "source":        "code2doc",
    }


def adapt_codesearchnet(example: dict) -> Optional[dict]:
    lang = normalize_language(example.get("language", ""))
    if not lang:
        return None
    doc  = (example.get("func_documentation_string") or "").strip()
    code = (example.get("func_code_string") or "").strip()
    if not doc or not code:
        return None
    return {
        "code":          code,
        "target_doc":    doc,
        "language":      lang,
        "style":         infer_style(doc, lang),
        "quality_score": 0.55,   # base trust for CSN
        "source":        "codesearchnet",
    }


def adapt_codexglue(example: dict, language: str) -> Optional[dict]:
    lang = normalize_language(language)
    if not lang:
        return None
    code = (example.get("code") or "").strip()
    doc  = (example.get("docstring") or "").strip()
    if not code or not doc:
        return None
    return {
        "code":          code,
        "target_doc":    doc,
        "language":      lang,
        "style":         infer_style(doc, lang),
        "quality_score": 0.80,
        "source":        "codexglue",
    }


def adapt_docstring_corpus(example: dict) -> Optional[dict]:
    code = (example.get("function") or example.get("code") or "").strip()
    doc  = (example.get("docstring") or "").strip()
    if not code or not doc:
        return None
    return {
        "code":          code,
        "target_doc":    doc,
        "language":      "python",
        "style":         infer_style(doc, "python"),
        "quality_score": 0.65,
        "source":        "docstring_corpus",
    }


def adapt_self_oss_instruct(example: dict) -> Optional[dict]:
    """self-oss-instruct stores prompt + response; extract code from prompt."""
    prompt   = example.get("prompt", "")
    response = (example.get("response") or "").strip()
    code = extract_code_block(prompt)
    if not code or not response:
        return None
    # Only keep if response looks like documentation (has words, not just code)
    if len(response.split()) < MIN_DOC_TOKENS:
        return None
    return {
        "code":          code,
        "target_doc":    response,
        "language":      "python",
        "style":         infer_style(response, "python"),
        "quality_score": 0.85,
        "source":        "self_oss_instruct",
    }


print("✅ Dataset adapters defined")

## Cell 5 — Load Tier 1: code2doc

In [ ]:
print("📦 Loading code2doc...")

tier1_samples = []

try:
    raw_code2doc = load_dataset(
        "kaanrkaraman/code2doc",
        split="train",
        cache_dir=str(CACHE_DIR),
    )
    print(f"   Raw size: {len(raw_code2doc):,}")

    kept = 0
    for ex in raw_code2doc:
        if kept >= CAPS["code2doc"]:
            break
        sample = adapt_code2doc(ex)
        if passes_quality_filter(sample) and not is_duplicate(sample["code"]):
            tier1_samples.append(sample)
            kept += 1

    print(f"   ✅ Kept: {kept:,}  (dropped {len(raw_code2doc) - kept:,})")

    # Free memory
    del raw_code2doc

except Exception as e:
    print(f"   ⚠️  code2doc failed: {e}")
    print("   Continuing without it.")

## Cell 6 — Load Tier 1: CodeXGLUE

In [ ]:
print("📦 Loading CodeXGLUE (code_x_glue_ct_code_to_text)...")

CODEXGLUE_LANGS = ["python", "java", "javascript"]
PER_LANG_CAP = CAPS["codexglue"] // len(CODEXGLUE_LANGS)  # 10K each

for lang in CODEXGLUE_LANGS:
    print(f"   Loading {lang}...")
    try:
        raw = load_dataset(
            "google/code_x_glue_ct_code_to_text",
            lang,
            split="train",
            cache_dir=str(CACHE_DIR),
        )
        kept = 0
        for ex in raw:
            if kept >= PER_LANG_CAP:
                break
            sample = adapt_codexglue(ex, lang)
            if passes_quality_filter(sample) and not is_duplicate(sample["code"]):
                tier1_samples.append(sample)
                kept += 1
        print(f"   ✅ {lang}: kept {kept:,}")
        del raw

    except Exception as e:
        print(f"   ⚠️  CodeXGLUE/{lang} failed: {e}")

print(f"\n📊 Tier 1 total so far: {len(tier1_samples):,}")

## Cell 7 — Load Tier 1: self-oss-instruct

In [ ]:
print("📦 Loading self-oss-instruct-sc2-exec-filter-50k...")

try:
    raw_oss = load_dataset(
        "bigcode/self-oss-instruct-sc2-exec-filter-50k",
        split="train",
        cache_dir=str(CACHE_DIR),
    )
    print(f"   Raw size: {len(raw_oss):,}")

    kept = 0
    for ex in raw_oss:
        if kept >= CAPS["self_oss_instruct"]:
            break
        sample = adapt_self_oss_instruct(ex)
        if passes_quality_filter(sample) and not is_duplicate(sample["code"]):
            tier1_samples.append(sample)
            kept += 1

    print(f"   ✅ Kept: {kept:,}  (dropped {len(raw_oss) - kept:,})")
    del raw_oss

except Exception as e:
    print(f"   ⚠️  self-oss-instruct failed: {e}")
    print("   Continuing without it.")

print(f"\n📊 Tier 1 FINAL total: {len(tier1_samples):,}")

# Language distribution snapshot
tier1_lang_dist = Counter(s["language"] for s in tier1_samples)
print(f"   Language distribution: {dict(tier1_lang_dist)}")

## Cell 8 — Load Tier 2: CodeSearchNet (streamed)

In [ ]:
print("📦 Streaming CodeSearchNet (20K per language)...")

tier2_samples = []
CSN_LANGS     = ["python", "java", "javascript"]
CSN_PER_LANG  = CAPS["codesearchnet"] // len(CSN_LANGS)  # 20K each

for lang in CSN_LANGS:
    print(f"   Streaming {lang}...")
    try:
        streamed = load_dataset(
            "code_search_net",
            lang,
            split="train",
            streaming=True,
            cache_dir=str(CACHE_DIR),
            trust_remote_code=True,
        )

        kept = 0
        scanned = 0
        for ex in streamed:
            scanned += 1
            if kept >= CSN_PER_LANG:
                break
            # Scan at most 4x the cap to avoid infinite loops on noisy datasets
            if scanned > CSN_PER_LANG * 4:
                break
            sample = adapt_codesearchnet(ex)
            if passes_quality_filter(sample) and not is_duplicate(sample["code"]):
                tier2_samples.append(sample)
                kept += 1

        print(f"   ✅ {lang}: kept {kept:,} / scanned {scanned:,}")

    except Exception as e:
        print(f"   ⚠️  CSN/{lang} failed: {e}")

print(f"\n📊 Tier 2 CSN total: {len(tier2_samples):,}")

## Cell 9 — Load Tier 2: docstring_corpus (streamed)

In [ ]:
print("📦 Streaming docstring_corpus...")

try:
    streamed_dc = load_dataset(
        "ncoop57/docstring_corpus",
        split="train",
        streaming=True,
        cache_dir=str(CACHE_DIR),
        trust_remote_code=True,
    )

    kept = 0
    scanned = 0
    cap = CAPS["docstring_corpus"]

    for ex in streamed_dc:
        scanned += 1
        if kept >= cap:
            break
        if scanned > cap * 4:
            break
        sample = adapt_docstring_corpus(ex)
        if passes_quality_filter(sample) and not is_duplicate(sample["code"]):
            tier2_samples.append(sample)
            kept += 1

    print(f"   ✅ Kept: {kept:,} / scanned {scanned:,}")

except Exception as e:
    print(f"   ⚠️  docstring_corpus failed: {e}")
    print("   Continuing without it.")

print(f"\n📊 Tier 2 FINAL total: {len(tier2_samples):,}")
tier2_lang_dist = Counter(s["language"] for s in tier2_samples)
print(f"   Language distribution: {dict(tier2_lang_dist)}")

## Cell 10 — Merge, Balance & Final Deduplication

In [ ]:
print("🔀 Merging and balancing...")

# Combine all tiers
all_samples = tier1_samples + tier2_samples
print(f"   Combined raw total: {len(all_samples):,}")

# Shuffle
random.shuffle(all_samples)

# Group by language
by_lang = defaultdict(list)
for s in all_samples:
    by_lang[s["language"]].append(s)

print("   Counts by language before balancing:")
for lang, items in by_lang.items():
    print(f"     {lang}: {len(items):,}")

# ── Stratified sampling to hit target language ratios ─────────────────────────
balanced = []
for lang, target_ratio in LANGUAGE_TARGET.items():
    target_n = int(FINAL_DATASET_SIZE * target_ratio)
    available = by_lang.get(lang, [])

    # Sort available samples: Tier 1 first (higher quality), then by quality_score
    TIER1_SOURCES = {"code2doc", "codexglue", "self_oss_instruct"}
    tier1_items = [s for s in available if s["source"] in TIER1_SOURCES]
    tier2_items = [s for s in available if s["source"] not in TIER1_SOURCES]

    # Sort each group by quality_score descending
    tier1_items.sort(key=lambda x: x["quality_score"], reverse=True)
    tier2_items.sort(key=lambda x: x["quality_score"], reverse=True)

    # Fill quota: Tier 1 first, then Tier 2
    selected = (tier1_items + tier2_items)[:target_n]
    balanced.extend(selected)

    print(f"   {lang}: target={target_n:,} | available={len(available):,} | selected={len(selected):,}")

# Final shuffle
random.shuffle(balanced)

print(f"\n📊 Final balanced dataset: {len(balanced):,} samples")
final_lang_dist = Counter(s["language"] for s in balanced)
final_source_dist = Counter(s["source"] for s in balanced)
final_style_dist = Counter(s["style"] for s in balanced)

print("\n   Language distribution:")
for lang, count in final_lang_dist.most_common():
    pct = count / len(balanced) * 100
    print(f"     {lang:15s}: {count:6,}  ({pct:.1f}%)")

print("\n   Source distribution:")
for src, count in final_source_dist.most_common():
    pct = count / len(balanced) * 100
    print(f"     {src:25s}: {count:6,}  ({pct:.1f}%)")

print("\n   Style distribution:")
for style, count in final_style_dist.most_common():
    pct = count / len(balanced) * 100
    print(f"     {style:20s}: {count:6,}  ({pct:.1f}%)")

## Cell 11 — Format to Qwen2.5 Chat Template

In [ ]:
print("💬 Loading Qwen2.5-Coder tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=str(CACHE_DIR),
    trust_remote_code=True,
)

SYSTEM_PROMPTS = {
    "python": (
        "You are a Python documentation assistant. Given Python source code, "
        "generate accurate, well-structured documentation following the specified "
        "style. Include a concise summary, parameter descriptions with types where "
        "inferable, and return value descriptions. Be precise and factual."
    ),
    "java": (
        "You are a Java documentation assistant. Given Java source code, generate "
        "accurate Javadoc-style documentation. Include a concise summary, @param "
        "tags for all parameters, @return tag if applicable, and @throws if "
        "exceptions are declared. Be precise and factual."
    ),
    "javascript": (
        "You are a JavaScript documentation assistant. Given JavaScript or "
        "TypeScript source code, generate accurate JSDoc-style documentation. "
        "Include a concise summary, @param tags with types, and @returns tag "
        "where applicable. Be precise and factual."
    ),
}


def format_to_chat(sample: dict) -> Optional[dict]:
    """Format a unified schema sample into Qwen2.5 chat template."""
    lang  = sample["language"]
    style = sample["style"]
    code  = sample["code"]
    doc   = sample["target_doc"]

    system = SYSTEM_PROMPTS.get(lang, SYSTEM_PROMPTS["python"])

    user_content = (
        f"Language: {lang}\n"
        f"Documentation style: {style}\n\n"
        f"```{lang}\n{code}\n```\n\n"
        "Generate documentation for the above code."
    )

    messages = [
        {"role": "system",    "content": system},
        {"role": "user",      "content": user_content},
        {"role": "assistant", "content": doc},
    ]

    # Apply chat template (includes EOS at end of assistant turn)
    try:
        formatted = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
    except Exception:
        return None

    # Check sequence length
    token_len = len(tokenizer.encode(formatted))
    if token_len > MAX_SEQ_LENGTH:
        return None   # skip — will overflow context window during training

    return {
        "text":          formatted,
        "messages":      messages,
        "language":      lang,
        "style":         style,
        "quality_score": sample["quality_score"],
        "source":        sample["source"],
        "token_length":  token_len,
    }


print("💬 Formatting samples to chat template...")
formatted_samples = []
skipped_length = 0

for i, sample in enumerate(balanced):
    if i % 10_000 == 0 and i > 0:
        print(f"   Processed {i:,} / {len(balanced):,}...")
    result = format_to_chat(sample)
    if result is None:
        skipped_length += 1
    else:
        formatted_samples.append(result)

print(f"\n✅ Formatted: {len(formatted_samples):,}")
print(f"   Skipped (too long): {skipped_length:,}")

# Token length stats
lengths = [s["token_length"] for s in formatted_samples]
print(f"\n   Token length stats:")
print(f"     Min    : {min(lengths)}")
print(f"     Max    : {max(lengths)}")
print(f"     Mean   : {sum(lengths)/len(lengths):.0f}")
print(f"     Median : {sorted(lengths)[len(lengths)//2]}")

## Cell 12 — Train / Validation Split

In [ ]:
print("✂️  Creating train/validation split...")

random.shuffle(formatted_samples)

VALIDATION_RATIO = 0.05   # 5% validation — enough for monitoring, not wasteful
val_size = int(len(formatted_samples) * VALIDATION_RATIO)

val_samples   = formatted_samples[:val_size]
train_samples = formatted_samples[val_size:]

print(f"   Train : {len(train_samples):,}")
print(f"   Val   : {len(val_samples):,}")

# Verify no language/source collapse in val set
val_lang_dist = Counter(s["language"] for s in val_samples)
print(f"   Val language dist: {dict(val_lang_dist)}")

## Cell 13 — Save Outputs

In [ ]:
print("💾 Saving datasets...")

# ── HuggingFace Dataset format (preferred for training with TRL/SFTTrainer) ───
train_hf = Dataset.from_list(train_samples)
val_hf   = Dataset.from_list(val_samples)

dataset_dict = DatasetDict({
    "train":      train_hf,
    "validation": val_hf,
})

dataset_dict.save_to_disk(str(OUTPUT_DIR / "hf_dataset"))
print(f"   ✅ HF Dataset saved to: {OUTPUT_DIR / 'hf_dataset'}")

# ── JSONL format (useful for manual inspection + backup) ──────────────────────
for split_name, split_data in [("train", train_samples), ("validation", val_samples)]:
    jsonl_path = OUTPUT_DIR / f"{split_name}.jsonl"
    with open(jsonl_path, "w", encoding="utf-8") as f:
        for sample in split_data:
            f.write(json.dumps(sample, ensure_ascii=False) + "\n")
    print(f"   ✅ {split_name}.jsonl saved ({jsonl_path.stat().st_size / 1e6:.1f} MB)")

# ── Metadata / stats file ─────────────────────────────────────────────────────
stats = {
    "total_samples":     len(formatted_samples),
    "train_samples":     len(train_samples),
    "val_samples":       len(val_samples),
    "language_dist":     dict(Counter(s["language"] for s in formatted_samples)),
    "source_dist":       dict(Counter(s["source"] for s in formatted_samples)),
    "style_dist":        dict(Counter(s["style"] for s in formatted_samples)),
    "token_length_mean": int(sum(s["token_length"] for s in formatted_samples) / len(formatted_samples)),
    "token_length_max":  max(s["token_length"] for s in formatted_samples),
    "model_id":          MODEL_ID,
    "seed":              SEED,
    "quality_threshold": MIN_QUALITY,
    "max_seq_length":    MAX_SEQ_LENGTH,
}

stats_path = OUTPUT_DIR / "dataset_stats.json"
with open(stats_path, "w") as f:
    json.dump(stats, f, indent=2)
print(f"   ✅ Stats saved to: {stats_path}")

# ── Disk usage summary ────────────────────────────────────────────────────────
import shutil
disk_used = shutil.disk_usage("/kaggle/working")
print(f"\n   Disk used: {disk_used.used / 1e9:.2f} GB / {disk_used.total / 1e9:.2f} GB")

## Cell 14 — Sample Inspection
Visually verify a few formatted samples before trusting the pipeline.

In [ ]:
print("🔍 Sample inspection — one per language\n")
print("=" * 80)

shown = set()
for sample in formatted_samples:
    lang = sample["language"]
    if lang in shown:
        continue
    shown.add(lang)

    print(f"\n{'─'*80}")
    print(f"  Language : {lang}")
    print(f"  Style    : {sample['style']}")
    print(f"  Source   : {sample['source']}")
    print(f"  Quality  : {sample['quality_score']:.2f}")
    print(f"  Tokens   : {sample['token_length']}")
    print(f"{'─'*80}")
    # Show just the user + assistant turns for readability
    msgs = sample["messages"]
    print(f"[USER]\n{msgs[1]['content'][:400]}..." if len(msgs[1]['content']) > 400 else f"[USER]\n{msgs[1]['content']}")
    print(f"\n[ASSISTANT]\n{msgs[2]['content'][:400]}..." if len(msgs[2]['content']) > 400 else f"\n[ASSISTANT]\n{msgs[2]['content']}")

    if len(shown) == 3:
        break

print(f"\n{'='*80}")

## Cell 15 — Final Summary

In [ ]:
print("\n" + "═" * 60)
print("  DATASET CREATION COMPLETE")
print("═" * 60)
print(f"  Train samples    : {len(train_samples):>10,}")
print(f"  Val samples      : {len(val_samples):>10,}")
print(f"  Total            : {len(formatted_samples):>10,}")
print()
print("  Language split:")
for lang, count in Counter(s['language'] for s in formatted_samples).most_common():
    bar = "█" * int(count / len(formatted_samples) * 40)
    print(f"    {lang:12s} {bar} {count:,}")
print()
print("  Source breakdown:")
for src, count in Counter(s['source'] for s in formatted_samples).most_common():
    print(f"    {src:25s}: {count:,}")
print()
print("  Outputs saved to:")
for f in sorted(OUTPUT_DIR.rglob("*")):
    if f.is_file():
        print(f"    {f.relative_to('/kaggle/working')}  ({f.stat().st_size / 1e6:.1f} MB)")
print()
print("  Next step: Upload /kaggle/working/slm_docgen_dataset")
print("  as a Kaggle Dataset to persist between sessions.")
print("═" * 60)

---
## ⬆️ Persisting Output to a Kaggle Dataset

After running this notebook:

1. In the right sidebar click **Data** → **Upload** → **New Dataset**
2. Select the `/kaggle/working/slm_docgen_dataset/` folder
3. Name it `slm-docgen-dataset` and set visibility to **Private**
4. In your training notebook, attach it under **Add Data** → **Your Datasets**
5. Access it at `/kaggle/input/slm-docgen-dataset/`

Then in your training notebook load with:
```python
from datasets import load_from_disk
dataset = load_from_disk("/kaggle/input/slm-docgen-dataset/hf_dataset")
train_dataset = dataset["train"]
val_dataset   = dataset["validation"]
```